# English Text Summarization
Paste any English text and get a concise summary powered by BART.

In [1]:
# ── Config ──────────────────────────────────────────────────────────────────
# Swap model to change summarization style:
#   News / articles  : "facebook/bart-large-cnn"          (~1.6 GB)
#   Conversations    : "philschmid/bart-large-cnn-samsum" (~1.6 GB)
#   Lightweight      : "sshleifer/distilbart-cnn-12-6"    (~1.2 GB)

MODEL_NAME  = "facebook/bart-large-cnn"
MIN_TOKENS  = 30     # minimum summary length
MAX_TOKENS  = 160    # maximum summary length

In [2]:
# ── Setup ────────────────────────────────────────────────────────────────────
# !uv pip install transformers torch ipywidgets

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import ipywidgets as widgets
from IPython.display import display, HTML

print(f"Loading model: {MODEL_NAME} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print("Ready.")

Loading model: facebook/bart-large-cnn …


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Ready.


In [3]:
# ── Widget ───────────────────────────────────────────────────────────────────
display(HTML("""
<style>
  .sum-card {
    background:#1e1e2e; border-radius:12px; padding:20px 24px;
    max-width:720px; font-family:'Inter',sans-serif;
    box-shadow:0 4px 20px rgba(0,0,0,.4);
  }
  .sum-card h3 { margin:0 0 4px; color:#cdd6f4; font-size:1.1rem; }
  .sum-card p  { margin:0 0 16px; color:#6c7086; font-size:.8rem; }
  .badge {
    display:inline-block; padding:2px 10px; border-radius:20px;
    font-size:.72rem; font-weight:700; letter-spacing:.05em;
    text-transform:uppercase; margin-bottom:5px;
  }
  .badge-in  { background:#313244; color:#89dceb; }
  .badge-out { background:#313244; color:#a6e3a1; }
  .out-box {
    background:#181825; border:1px solid #45475a; border-radius:8px;
    padding:12px 14px; color:#cdd6f4; min-height:72px;
    font-size:1rem; white-space:pre-wrap; line-height:1.6;
  }
  .placeholder { color:#45475a; font-style:italic; }
  .word-count  { color:#6c7086; font-size:.75rem; margin-top:3px; }
</style>
"""))

header = widgets.HTML(
    f'<div class="sum-card"><h3>📝 Text Summarization</h3>'
    f'<p>{MODEL_NAME}</p></div>'
)

badge_in  = widgets.HTML('<span class="badge badge-in">Input</span>')
badge_out = widgets.HTML('<span class="badge badge-out">Summary</span>')

input_box = widgets.Textarea(
    placeholder="Paste English text here…",
    layout=widgets.Layout(width="100%", height="180px"),
)
word_count = widgets.HTML('<div class="word-count">0 words</div>')

btn_summarize = widgets.Button(
    description="Summarize",
    layout=widgets.Layout(width="110px", height="34px"),
    style=widgets.ButtonStyle(button_color="#89b4fa", font_weight="bold"),
)
btn_clear = widgets.Button(
    description="Clear",
    layout=widgets.Layout(width="72px", height="34px"),
    style=widgets.ButtonStyle(button_color="#45475a"),
)
status = widgets.Label(value="")
output = widgets.HTML('<div class="out-box placeholder">Summary appears here.</div>')

def update_word_count(change):
    n = len(change["new"].split())
    word_count.value = f'<div class="word-count">{n} word{"s" if n != 1 else ""}</div>'

input_box.observe(update_word_count, names="value")

def on_summarize(_):
    text = input_box.value.strip()
    if not text:
        status.value = "⚠️ Paste some text first."
        return
    if len(text.split()) < 30:
        status.value = "⚠️ Text is too short to summarize."
        return
    status.value = "Summarizing…"
    btn_summarize.disabled = True
    try:
        inputs  = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
        outputs = model.generate(
            **inputs,
            min_length=MIN_TOKENS,
            max_length=MAX_TOKENS,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True,
        )
        result = tokenizer.decode(outputs[0], skip_special_tokens=True)
        output.value = f'<div class="out-box">{result}</div>'
        status.value = "✓ Done"
    except Exception as e:
        output.value = f'<div class="out-box" style="color:#f38ba8">Error: {e}</div>'
        status.value = ""
    finally:
        btn_summarize.disabled = False

def on_clear(_):
    input_box.value = ""
    output.value = '<div class="out-box placeholder">Summary appears here.</div>'
    status.value = ""
    word_count.value = '<div class="word-count">0 words</div>'

btn_summarize.on_click(on_summarize)
btn_clear.on_click(on_clear)

display(widgets.VBox([
    header,
    badge_in,
    input_box,
    widgets.HBox([btn_summarize, btn_clear, status],
                 layout=widgets.Layout(align_items="center", gap="8px", margin="4px 0")),
    word_count,
    badge_out,
    output,
], layout=widgets.Layout(max_width="720px", gap="3px")))